In [1]:
from __future__ import print_function, division
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torch
#from test_bottleneck import *

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


class Conv_block(nn.Module):
    """
    Convolution Block #卷积块
    """
    def __init__(self, in_ch, out_ch):
        super(Conv_block, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            #nn.GroupNorm(16,out_ch),
            nn.Hardswish(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            #nn.GroupNorm(16,out_ch),
            nn.Hardswish(inplace=True))
        self.conv_1x1_output = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1)

    def forward(self, x):

        y = self.conv(x)
        x = self.conv_1x1_output(x)
        return x+y


class Dense_layer(nn.Sequential):
    def __init__(self,in_ch,bottleneck_size,growth_rate):
    
        super(Dense_layer,self).__init__()
        
        self.add_module('norm1', nn.BatchNorm2d(in_ch)),
        self.add_module('relu1', nn.Hardswish(inplace=True)),
        self.add_module('conv1', nn.Conv2d(in_ch, bottleneck_size, kernel_size=1, stride=1, padding=0, bias=True)),
        self.add_module('norm2', nn.BatchNorm2d(bottleneck_size)),
        self.add_module('relu2', nn.Hardswish(inplace=True)),
        self.add_module('conv2', nn.Conv2d(bottleneck_size, growth_rate, kernel_size=3, stride=1, padding=1, bias=True)),

    def forward(self, x):
        new_features = super(Dense_layer, self).forward(x)
        return torch.cat([x, new_features], 1)

class Dense_block(nn.Sequential):
            
    def __init__(self,in_channels,layer_counts,growth_rate):
        super(Dense_block,self).__init__()
        for i in range(layer_counts):
            bottleneck_size = 4*growth_rate
            layer = Dense_layer(in_channels + i * growth_rate, bottleneck_size, growth_rate)
            self.add_module('denselayer%d' % (i + 1), layer)


class TransitionLayer(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        self.add_module('norm', nn.BatchNorm2d(in_channels))
        self.add_module('relu', nn.Hardswish(inplace=True))
        self.add_module('conv', nn.Conv2d(in_channels, out_channels,kernel_size=1, stride=1, bias=False))

class Conv_pooling(nn.Module):
    """
    # conv->pooling        #output_shape = (image_shape-filter_shape+2*padding)/stride + 1
    """
    def __init__(self, in_ch, out_ch):
        super(Conv_pooling, self).__init__()

        self.ConvPooling = nn.Sequential(
            #nn.Conv2d(in_ch, in_ch, kernel_size=2, stride=1, padding=1,dilation=2, bias=True),
            #nn.BatchNorm2d(in_ch),
            #nn.Hardswish(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2, padding=0, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True)
        )

    def forward(self, x):
        out=self.ConvPooling(x)
        return out


class ASPP(nn.Module):
    
    def __init__(self, in_channel):
        super(ASPP,self).__init__()
          
        self.atrous_block1 = nn.Conv2d(in_channel, in_channel, kernel_size=1, stride=1)
        self.atrous_block2 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=2, dilation=2)
        self.atrous_block3 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=3, dilation=3)
        self.atrous_block4 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=4, dilation=4)
        self.conv_1x1_output = nn.Conv2d(in_channel * 4, in_channel, kernel_size=1, stride=1)
        self.ConvPooling = nn.Sequential(
            #effective_size=(k−1)⋅d+1=(2−1)⋅2+1=3

            nn.Conv2d(in_channel, in_channel*2, kernel_size=2, stride=1, padding=1, dilation=2, bias=True),
            nn.BatchNorm2d(in_channel*2),
            nn.Hardswish(inplace=True),
            #nn.Conv2d(in_channel, in_channel, kernel_size=2, stride=2, padding=0, bias=True),
            nn.MaxPool2d(2,stride=2),
            nn.BatchNorm2d(in_channel*2),
            nn.Hardswish(inplace=True)
        )

    def forward(self, x):
        atrous_block1 = self.atrous_block1(x)
        atrous_block2 = self.atrous_block2(x)
        atrous_block3 = self.atrous_block3(x)
        atrous_block4 = self.atrous_block4(x)
        out = self.conv_1x1_output(torch.cat([ atrous_block1, atrous_block2, atrous_block3, atrous_block4], dim=1))
        out = self.ConvPooling(out+x)
        return out

class UpConv_block(nn.Module):
    """
    Convolution Block #卷积块
    """
    def __init__(self, in_ch, out_ch):
        super(UpConv_block, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear'),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True))

    def forward(self, x):

        y = self.conv(x)
        
        return y

class CatConv_block(nn.Module):
    """
    Convolution Block #卷积块
    """
    def __init__(self, in_ch, growth):
        super(CatConv_block, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch*growth, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True))
        
        self.Up=nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear'),
            nn.BatchNorm2d(in_ch*2),
            nn.Hardswish(inplace=True),
            nn.Conv2d(in_ch*2, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True))
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_ch*growth, in_ch*growth, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(in_ch*growth),
            nn.Hardswish(inplace=True),
            nn.Conv2d(in_ch*growth, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True)
            )

    def forward(self, x1, x2):
        
        x2 = self.Up(x2)
        x = torch.cat([x1,x2],dim=1)
        # conv2 256 256 64 conv3d for spatial + conv1d to make it 32 for summing
        # with conv1 that has chnanle reduction
        y = self.conv2(x)+self.conv1(x)
        
        return y

class Net(nn.Module):
    """
    UNet - Basic Implementation
    Paper : https://arxiv.org/abs/1505.04597
    """
    def __init__(self, in_ch=2, out_ch=1):
        super(Net, self).__init__()

        #self.swish=nn.SiLU(inplace=True)
        #self.swish=nn.Hardswish(inplace=True)
        #self.act=nn.Sigmoid()
        #self.Conv0=Conv_block(1,32)
        self.CP=Conv_pooling(1,32)
        self.Aspp1=ASPP(32) # 32 channels 
        self.Aspp2=ASPP(64)
        self.Aspp3=ASPP(128)
        self.Aspp4=ASPP(256)
        self.Aspp5=ASPP(512)
        #self.DP=nn.Dropout(0.2)
        #self.Conv00=Conv_block(1,32)
        self.conv1x1=nn.Conv2d(16, 1, kernel_size=1, stride=1)     
        self.Conv=Conv_block(32,16)
        
        self.Dense1=Dense_block(512,5,32)
        self.Dense2=Dense_block(256,5,16)
        self.Dense3=Dense_block(128,5,16)
        self.Dense4=Dense_block(64,5,16)
        self.Dense5=Dense_block(32,5,16)
        
        self.Trans1=TransitionLayer(512+5*32,512)
        self.Trans2=TransitionLayer(256+5*16,256)
        self.Trans3=TransitionLayer(128+5*16,128)
        self.Trans4=TransitionLayer(64+5*16,64)
        self.Trans5=TransitionLayer(32+5*16,32)
        
        self.CatConv01=CatConv_block(32,2)
        self.CatConv11=CatConv_block(64,2)
        self.CatConv21=CatConv_block(128,2)
        self.CatConv31=CatConv_block(256,2)
        self.CatConv41=CatConv_block(512,2)
        
        self.CatConv02=CatConv_block(512,2)
        self.CatConv12=CatConv_block(256,2)
        self.CatConv22=CatConv_block(128,2)
        self.CatConv32=CatConv_block(64,2)
        self.CatConv42=CatConv_block(32,2)
        
        #self.DP1=nn.Dropout2d(0.4,inplace=True)
        #self.DP2=nn.Dropout2d(0.4,inplace=True)
        
    def forward(self, x):
        
        # ----------Feature extraction part ------------

        Conv00=self.CP(x)  #1->32  512->256
        Conv10=self.Aspp1(Conv00) #32->64 channel  256->128 spatial
        Conv20=self.Aspp2(Conv10)   #64->128 128->64
        Conv30=self.Aspp3(Conv20)   #128->256  64->32
        Conv40=self.Aspp4(Conv30)   #256->512 32->16
        Conv50=self.Aspp5(Conv40)   #512->1024 16->8

        #-------------- feature fusion part ----------

        Conv01=self.CatConv01(Conv00,Conv10) #32->32  256->256
        Conv11=self.CatConv11(Conv10,Conv20) #64->64  128->128
        Conv21=self.CatConv21(Conv20,Conv30) #128->128  64->64
        Conv31=self.CatConv31(Conv30,Conv40) #256->256  32->32
        Conv41=self.CatConv41(Conv40,Conv50) #512->512  16->16
        
        Conv42=self.CatConv02(Conv41,Conv50) #512->512  16->16
        Conv42=self.Dense1(Conv42)
        Conv42=self.Trans1(Conv42)
        
        Conv32=self.CatConv12(Conv31,Conv42) #256->256  32->32
        Conv32=self.Dense2(Conv32)
        Conv32=self.Trans2(Conv32)
        
        Conv22=self.CatConv22(Conv21,Conv32) #128->128  64->64
        Conv22=self.Dense3(Conv22)
        Conv22=self.Trans3(Conv22)
        
        Conv12=self.CatConv32(Conv11,Conv22) #64->64  128->128
        Conv12=self.Dense4(Conv12)
        Conv12=self.Trans4(Conv12)
        
        Conv02=self.CatConv42(Conv01,Conv12) #32->32  256->256
        Conv02=self.Dense5(Conv02)
        Conv02=self.Trans5(Conv02)

        Conv0=self.Conv(Conv02) #32->16 256->256
        out=self.conv1x1(Conv0) #16->1 256->256

        return out

## preparing dataset

In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader  
from torchvision import transforms
from PIL import Image

class ImageDataset(Dataset):
    def __init__(self, inputs_path, target_path, train_target_size=(512, 512) , output_target_size = (256 ,256)):
        self.inputs_path = inputs_path          
        self.target_path = target_path        
        self.target_size = train_target_size        
        self.output_size = output_target_size        
        self.transform = transforms.ToTensor()

        # load single target once
        t_img = Image.open(self.target_path).convert('L')
        t_img = t_img.resize(self.output_size, Image.LANCZOS)
        self.target_tensor = self.transform(t_img)

    def __len__(self):
        return len(self.inputs_path)           

    def __getitem__(self, index):
        x_path = self.inputs_path[index]

        x_image = Image.open(x_path).convert('L')
        x_image = x_image.resize(self.target_size, Image.LANCZOS)

        x_tensor = self.transform(x_image)
        y_tensor = self.target_tensor          

        return x_tensor, y_tensor

### preparing data and model

In [3]:
inputs_path = [os.path.join('train_images' , img) for img in os.listdir('train_images')]
target_path = os.path.join('quantom_dataset/target/ground_truth.png')
len(inputs_path) , target_path

(9720, 'quantom_dataset/target/ground_truth.png')

In [4]:
dataset =  ImageDataset(inputs_path , target_path)
len(dataset)

9720

In [5]:
# single_image_path = [inputs_path[2]]  
# dataset_single = ImageDataset(single_image_path, target_path)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)

# print(f"Training on: {single_image_path[0]}")
# print(f"Dataset size: {len(dataset_single)}")

for b , i in dataloader: 
    print(b.shape)
    print(i.shape)
    break

torch.Size([8, 1, 512, 512])
torch.Size([8, 1, 256, 256])


In [9]:
# First, split the dataset into train (90%) and test (10%)
from torch.utils.data import random_split

# Calculate split sizes
total_size = len(dataset)
test_size = int(0.02* total_size)  # 10% for test
train_size = total_size - test_size  # 90% for train

# Split the dataset
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create separate dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

Train samples: 9526, Test samples: 194


In [ ]:
# dataloader = DataLoader(dataset , batch_size=1, shuffle=False)
# single_data_loader = DataLoader(dataset_single, batch_size=1, shuffle=False)

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Net(in_ch=1 , out_ch=1).to(device)
model , device

(Net(
   (CP): Conv_pooling(
     (ConvPooling): Sequential(
       (0): Conv2d(1, 32, kernel_size=(2, 2), stride=(2, 2))
       (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (2): Hardswish()
     )
   )
   (Aspp1): ASPP(
     (atrous_block1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
     (atrous_block2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
     (atrous_block3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(3, 3), dilation=(3, 3))
     (atrous_block4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(4, 4), dilation=(4, 4))
     (conv_1x1_output): Conv2d(128, 32, kernel_size=(1, 1), stride=(1, 1))
     (ConvPooling): Sequential(
       (0): Conv2d(32, 64, kernel_size=(2, 2), stride=(1, 1), padding=(1, 1), dilation=(2, 2))
       (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       (2): Hardswish()
       (3): MaxPool2d(kerne

In [3]:
from torchinfo import summary
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Net(in_ch=1, out_ch=1).to(device)

# Input size: (batch_size, channels, height, width)
summary(model, input_size=(1, 1, 512, 512), col_names=["input_size", "output_size", "num_params"])


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Net                                      [1, 1, 512, 512]          [1, 1, 256, 256]          --
├─Conv_pooling: 1-1                      [1, 1, 512, 512]          [1, 32, 256, 256]         --
│    └─Sequential: 2-1                   [1, 1, 512, 512]          [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                  [1, 1, 512, 512]          [1, 32, 256, 256]         160
│    │    └─BatchNorm2d: 3-2             [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─Hardswish: 3-3               [1, 32, 256, 256]         [1, 32, 256, 256]         --
├─ASPP: 1-2                              [1, 32, 256, 256]         [1, 64, 128, 128]         --
│    └─Conv2d: 2-2                       [1, 32, 256, 256]         [1, 32, 256, 256]         1,056
│    └─Conv2d: 2-3                       [1, 32, 256, 256]         [1, 32, 256, 256]         9,248
│    └─Conv2d: 2-4          

In [4]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

Total parameters: 44,781,537


### starting training

In [12]:
optimizer = torch.optim.Adam(model.parameters() , lr=0.001)
criterion_l1 = nn.L1Loss()
loss_history = [] 

In [ ]:
model.train()
number_of_epochs = 500
from tqdm import tqdm
import numpy as np

best_val_loss = float('inf')
patience = 20
patience_counter = 0

criterion_l1 = nn.L1Loss()
train_loss_history = []
val_loss_history = []

for epoch in tqdm(range(number_of_epochs)):
    # ------- TRAIN -------
    model.train()
    current_epoch_train_loss = []
    for x, y in train_dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion_l1(output, y)
        current_epoch_train_loss.append(loss.item())
        loss.backward()
        optimizer.step()

    mean_train_loss = np.mean(current_epoch_train_loss)
    train_loss_history.append(mean_train_loss)

    # ------- VALIDATION -------
    model.eval()
    current_epoch_val_loss = []
    with torch.no_grad():
        for x_val, y_val in test_dataloader:
            x_val, y_val = x_val.to(device), y_val.to(device)
            output_val = model(x_val)
            val_loss = criterion_l1(output_val, y_val)
            current_epoch_val_loss.append(val_loss.item())

    mean_val_loss = np.mean(current_epoch_val_loss)
    val_loss_history.append(mean_val_loss)

    print(f"Epoch {epoch+1}: train_loss={mean_train_loss:.4f}, val_loss={mean_val_loss:.4f}")

    # early stopping & checkpoint based on validation loss
    if mean_val_loss < best_val_loss:
        best_val_loss = mean_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'trained_model.pth')
        torch.save({
            'model_state_dict': model.state_dict(),
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
        }, 'checkpoint.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

In [ ]:
import torch

device = torch.device('cpu')

state_dict = torch.load('trained_model.pth', map_location=device)

loss_history = []
try:
    ckpt = torch.load('checkpoint.pth', map_location=device, weights_only=False)
    loss_history = ckpt.get('loss_history', [])
    print(f"Loss history: {len(loss_history)} epochs")
except FileNotFoundError:
    print("No checkpoint.pth; only weights loaded.")

model = Net(in_ch=1, out_ch=1)
model.load_state_dict(state_dict)

model = model.to(device)

In [ ]:
import torch
import matplotlib.pyplot as plt

checkpoint = torch.load('checkpoint.pth', map_location='cpu', weights_only=False)

# Handle both old and new formats
train_loss_history = checkpoint.get('train_loss_history', None)
val_loss_history   = checkpoint.get('val_loss_history', None)

# If you only saved a single 'loss_history', treat it as train
if train_loss_history is None and 'loss_history' in checkpoint:
    train_loss_history = checkpoint['loss_history']

plt.figure(figsize=(8, 5))

if train_loss_history is not None:
    plt.plot(train_loss_history, label='Train loss')

if val_loss_history is not None:
    plt.plot(val_loss_history, label='Validation loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training / Validation Loss')
plt.grid(True)
plt.legend()
plt.show()

### inferencing the model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
import cv2 as cv 
model.eval()

input_img = Image.open('train_images/translate_train(2)_0_12.png').convert('L')
input_tensor = transforms.ToTensor()(input_img.resize((512, 512), Image.LANCZOS)).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_tensor)
    pred_np = output[0, 0].cpu().numpy()
    pred_np = np.clip(pred_np, 0, 1)

gt_img = Image.open('quantom_dataset/target/ground_truth.png')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(input_img)
axes[0].set_title('Input')
axes[0].axis('off')

axes[1].imshow(cv.resize(pred_np , (419 , 171)), cmap='viridis')
axes[1].set_title('Predicted')
axes[1].axis('off')

axes[2].imshow(gt_img)
axes[2].set_title('Ground Truth')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# After inference
print(f"Output shape: {output.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Compare actual values
print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")
print(f"Target range: [{y.min():.3f}, {y.max():.3f}]")

In [ ]:
# Compare first epoch loss vs last epoch loss
print(f"First epoch loss: {loss_history[0]:.6f}")
print(f"Last epoch loss: {loss_history[-1]:.6f}")

In [ ]:
# import numpy as np
# from PIL import Image
# import matplotlib.pyplot as plt
# from torchvision import transforms
# import os

# # Set device to CPU
# device = torch.device('cpu')
# print(f"Using device: {device}")

# # Initialize model
# model = Net(in_ch=1, out_ch=1).to(device)
# print("Model initialized!")

# # Load and preprocess a single image
# def load_image(image_path, target_size=(512, 512)):
#     """
#     Load an image and convert it to grayscale tensor
#     If image_path is None, create a test image
#     """
#     if image_path is None or not os.path.exists(image_path):
#         # Create a test image if no path provided
#         print("No image path provided, creating a test image...")
#         test_img = np.random.rand(512, 512) * 255
#         img = Image.fromarray(test_img.astype(np.uint8)).convert('L')
#     else:
#         img = Image.open(image_path).convert('L')
#         print(f"Loaded image from: {image_path}")
#         print(f"Original image size: {img.size}")
    
#     # Resize to target size
#     img = img.resize(target_size, Image.LANCZOS)
    
#     # Convert to tensor: [1, 1, H, W] with values in [0, 1]
#     transform = transforms.Compose([
#         transforms.ToTensor()
#     ])
#     tensor = transform(img).unsqueeze(0)  # Add batch dimension
    
#     return tensor, img

# # You can provide your image path here, or set to None to use a test image
# image_path = None  # Change this to your image path, e.g., "path/to/your/image.jpg"

# # Load image
# input_tensor, original_img = load_image(image_path, target_size=(512, 512))
# input_tensor = input_tensor.to(device)

# print(f"Input tensor shape: {input_tensor.shape}")
# print(f"Input tensor range: [{input_tensor.min():.3f}, {input_tensor.max():.3f}]")


In [ ]:
# # Training setup
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# # Training parameters
# num_epochs = 100
# print_interval = 10

# # Store loss history
# loss_history = []

# # Training loop
# print("\nStarting training...")
# print(f"Training for {num_epochs} epochs on CPU")
# print("-" * 50)

# model.train()
# for epoch in range(num_epochs):
#     optimizer.zero_grad()
    
#     # Forward pass
#     output = model(input_tensor)
    
#     # The model outputs 256x256, so we need to resize target to match
#     # Or we can resize the output back to 512x512
#     # Let's check the output size first
#     if epoch == 0:
#         print(f"Model output shape: {output.shape}")
#         print(f"Target shape: {input_tensor.shape}")
    
#     # Resize output to match input size if needed
#     if output.shape[2:] != input_tensor.shape[2:]:
#         output_resized = F.interpolate(output, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
#     else:
#         output_resized = output
    
#     # Calculate loss (reconstruction loss - trying to reconstruct the input)
#     loss = criterion(output_resized, input_tensor)
    
#     # Backward pass
#     loss.backward()
#     optimizer.step()
    
#     loss_history.append(loss.item())
    
#     # Print progress
#     if (epoch + 1) % print_interval == 0:
#         print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}")

# print("\nTraining completed!")
# print(f"Final loss: {loss_history[-1]:.6f}")


In [ ]:
# # ============================================
# # INFERENCE CODE - Run this after training
# # ============================================

# def inference(model, image_path, output_path=None, device='cpu'):
#     """
#     Run inference on a single image
    
#     Args:
#         model: Trained model
#         image_path: Path to input image
#         output_path: Path to save output image (optional)
#         device: Device to run on ('cpu' or 'cuda')
    
#     Returns:
#         reconstructed_image: PIL Image of the reconstruction
#     """
#     # Set model to evaluation mode
#     model.eval()
    
#     # Load and preprocess image
#     img = Image.open(image_path).convert('L')
#     original_size = img.size  # Save original size
    
#     # Resize to model's expected input size (512x512)
#     img_resized = img.resize((512, 512), Image.LANCZOS)
    
#     # Convert to tensor: [1, 1, H, W] with values in [0, 1]
#     transform = transforms.ToTensor()
#     input_tensor = transform(img_resized).unsqueeze(0).to(device)
    
#     print(f"Input image size: {original_size}")
#     print(f"Resized to: {img_resized.size}")
#     print(f"Input tensor shape: {input_tensor.shape}")
    
#     # Run inference
#     with torch.no_grad():
#         output = model(input_tensor)
#         print(f"Model output shape: {output.shape}")
        
#         # Resize output back to input size if needed
#         if output.shape[2:] != input_tensor.shape[2:]:
#             output_resized = F.interpolate(output, size=input_tensor.shape[2:], 
#                                          mode='bilinear', align_corners=False)
#         else:
#             output_resized = output
        
#         # Convert back to numpy array
#         output_np = output_resized[0, 0].cpu().numpy()
        
#         # Clip values to [0, 1] and convert to [0, 255]
#         output_np = np.clip(output_np, 0, 1)
#         output_np = (output_np * 255).astype(np.uint8)
        
#         # Convert to PIL Image
#         reconstructed_img = Image.fromarray(output_np, mode='L')
        
#         # Resize back to original image size if needed
#         if original_size != (512, 512):
#             reconstructed_img = reconstructed_img.resize(original_size, Image.LANCZOS)
#             print(f"Output resized back to: {original_size}")
    
#     # Save if output path provided
#     if output_path:
#         reconstructed_img.save(output_path)
#         print(f"Saved reconstructed image to: {output_path}")
    
#     return reconstructed_img

# # Example usage:
# # reconstructed = inference(model, "path/to/your/image.jpg", "output.jpg", device='cpu')
# # reconstructed.show()  # Display the image


In [ ]:
# # ============================================
# # EXAMPLE: Run inference on an image
# # ============================================

# # After training, use this to run inference on any image
# # Replace "your_image.jpg" with the path to your image

# input_image_path = "your_image.jpg"  # Change this to your image path
# output_image_path = "reconstructed_output.jpg"  # Where to save the result

# # Run inference
# try:
#     reconstructed = inference(model, input_image_path, output_image_path, device='cpu')
    
#     # Display both images side by side
#     fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
#     original = Image.open(input_image_path).convert('L')
#     axes[0].imshow(original, cmap='gray')
#     axes[0].set_title('Original Image')
#     axes[0].axis('off')
    
#     axes[1].imshow(reconstructed, cmap='gray')
#     axes[1].set_title('Reconstructed Image')
#     axes[1].axis('off')
    
#     plt.tight_layout()
#     plt.show()
    
# except FileNotFoundError:
#     print(f"Image not found: {input_image_path}")
#     print("Please update 'input_image_path' with a valid image path")
# except Exception as e:
#     print(f"Error: {e}")


In [ ]:
# # Visualization
# model.eval()
# with torch.no_grad():
#     reconstructed = model(input_tensor)
    
#     # Resize if needed
#     if reconstructed.shape[2:] != input_tensor.shape[2:]:
#         reconstructed_resized = F.interpolate(reconstructed, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
#     else:
#         reconstructed_resized = reconstructed

# # Convert tensors to numpy for visualization
# input_np = input_tensor[0, 0].cpu().numpy()
# reconstructed_np = reconstructed_resized[0, 0].cpu().numpy()

# # Calculate metrics
# mse = np.mean((input_np - reconstructed_np) ** 2)
# psnr = -10 * np.log10(mse) if mse > 0 else float('inf')

# print(f"\nReconstruction Metrics:")
# print(f"MSE: {mse:.6f}")
# print(f"PSNR: {psnr:.2f} dB")

# # Create visualization
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# # Original image
# axes[0].imshow(input_np, cmap='gray')
# axes[0].set_title('Original Image')
# axes[0].axis('off')

# # Reconstructed image
# axes[1].imshow(reconstructed_np, cmap='gray')
# axes[1].set_title(f'Reconstructed Image\n(MSE: {mse:.6f}, PSNR: {psnr:.2f} dB)')
# axes[1].axis('off')

# # Difference image
# diff = np.abs(input_np - reconstructed_np)
# axes[2].imshow(diff, cmap='hot')
# axes[2].set_title('Absolute Difference')
# axes[2].axis('off')

# plt.tight_layout()
# plt.show()

# # Plot loss history
# plt.figure(figsize=(10, 5))
# plt.plot(loss_history)
# plt.xlabel('Epoch')
# plt.ylabel('Loss (MSE)')
# plt.title('Training Loss History')
# plt.grid(True)
# plt.yscale('log')  # Use log scale for better visualization
# plt.show()

# print("\nVisualization complete!")
